<a href="https://colab.research.google.com/github/emsofjames2003/CB2330-exercises/blob/main/global_alignment_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Code: Global pairwise alignments

<!-- launch-badges -->
[![KTH JupyterHub](https://img.shields.io/badge/launch-KTH%20JupyterHub-F37626?logo=jupyter&logoColor=white)](https://193.10.159.40.nip.io/hub/user-redirect/git-pull?repo=https://github.com/statisticalbiotechnology/bibook&urlpath=lab/tree/bibook/bibook/alignment/pairwise/nw_code.ipynb&branch=main)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statisticalbiotechnology/bibook/blob/main/bibook/alignment/pairwise/nw_code.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/statisticalbiotechnology/bibook/main?labpath=bibook/alignment/pairwise/nw_code.ipynb)


We will now make use of our definitions of a Needleman-Wunsch alignment to see how the algorithm transforms to actual code. The sections below will walk you through how this is done.

:::{dropdown} Service functions for formatting and printing alignments
To facilitate the the reasoning in the subsequent cells, we first we define a couple of service functions that we will need later, for formating and printing alignments. It is not important that you understand what these functions do, for now.
:::

In [ ]:
import numpy as np

# Print 2 sequences on top of each other
def print_alignment(seqA,seqB):
    print(seqA)
    print(seqB) # prints on two seperate lines.

# Print the dynamic programming score matrix (S) together with the sequences.
# This helps visualize how the alignment scores are computed.
def print_dynamic(seqA,seqB,S,trace):

    seqA,seqB = "-" + seqA, "-" + seqB  # Add leading '-' for alignment with matrix indices
    m,n = len(seqA),len(seqB) # calculates total number of rows and columns including new gap
    print('{:^5}'.format(" "), end=""),  # Print header row: blank space in 5 character wide block. end="" stops from moving to next line
    for j in range(n):
        print('{:^5}'.format(seqB[j]), end="") # 5 character wide block, adds one base in, all on same line
    print()  # moves to next line once has cycled through all the bases in seqB.

    for i in range(m): # looping rows in seq A
        print ('{:^5}'.format(seqA[i]), end="")  # Print row label: prints first thing in seq A =  -
        for j in range(n): # inner loop: columns!
            print ('{:5.1f}'.format(S[i,j]), end="")  # computes the score for each box in the row
        print() # goes to next row, then loops back to for i in range(m): now i = 2 etc -> first base after - is added and keeps going.
    print()

# Format an alignment by inserting gaps ('-') in the sequences according to the trace matrix.
# Returns the aligned sequences as strings.
def format_alignment(seqA,seqB,S,trace):
    print("Best score: " + str(S[-1,-1]))  # Print the optimal alignment score: the one that is in the last row in the last column, so [-1, -1]
    outA,outB = "",""  # Output aligned sequences: two empty strings created
    i,j = len(seqA),len(seqB) # starting point of i, j is length of seqA & B -> initialized from the end of the sequence.
    while i>0 or j>0: # if the end of the sequence is larger than 0, which is top left of the matrix. then stops
        di,dj = trace[i,j]  # Get the direction from the trace matrix: di is vertical change, dj is horizontal change
        i += int(di) # adds directional pointers to di
        j += int(dj)
        if di == 0: # no vertical movement: has to be a gap
            outA = "-" + outA  # Insert gap in seqA : match/mismatch
        else:
            outA = seqA[i] + outA  # Add character from seqA
        if dj == 0: # no horizontal movement: insert
            outB = "-" + outB  # Insert gap in seqB
        else:
            outB = seqB[j] + outB  # Add character from seqB
    return outA,outB

## Scoring system for DNA sequences

We setup the scoring system we need for the alignment of DNA sequences. Here we use a score system where gaps score -2 and miss matches are scored -1 and matches get a score of 3. You should absolutely try to change the scoring function at some point to see how the values affect the alignments.

In [ ]:
# Returns the penalty for introducing a gap in the alignment.
def gap_penalty():
    return -2.0

# Returns the score for aligning two letters (could be nucleotides or amino acids).
# Matches get a positive score, mismatches a negative score, and gaps use the gap penalty.
def match_score(letterA,letterB):
    if letterA == '-' or letterB == '-':
        return gap_penalty()  # Gap penalty if either is a gap
    elif letterA == letterB:
        return 3.0  # Match score
    else:
        return -1.0  # Mismatch penalty

## Global alignments by Needleman-Wunsch

Here follows an implementation of the [Needleman-Wunsch](http://www.sciencedirect.com/science/article/pii/0022283670900574) pairwise alignment method.  We want to align two sequences $a=a_1,\ldots,a_{M}$ and $b=b_1,\ldots,b_{N}$.

As said in last chapter, the dynamic programming matrix $S$ is initiated as:
$S_{i0}=g \cdot i, \forall i,$
$S_{0j}=g \cdot j, \forall j$

This translates easily into two for loops filling in the upper row and the leftmost column.

### The trace matrix: keeping track of the optimal path

To reconstruct the optimal alignment after filling the score matrix $S$, we need to know which move (match/mismatch, insertion, or deletion) led to each cell's score. For this, we use a trace matrix. Each element in the trace matrix contains a pair of values $(di, dj)$ indicating the direction of the optimal step that led to the current cell:

- $(-1, -1)$: Diagonal move (match or mismatch, aligning $a_i$ with $b_j$)
- $(-1, 0)$: Upwards move (gap in sequence B, aligning $a_i$ with a gap)
- $(0, -1)$: Leftwards move (gap in sequence A, aligning $b_j$ with a gap)

By following these pointers from the bottom-right cell back to the top-left, we can reconstruct the optimal alignment by inserting gaps where needed. This process is called traceback.

Here we initiate pointers in the form of a trace matrix. Each element in the trace matrix contains the difference in index between the current cell and the optimal step that lead to the current cell.

In [ ]:
# Initialize the dynamic programming matrices for global alignment.
# S: score matrix, trace: matrix to keep track of the optimal path.
def initiate_global_dp(m,n): # number of rows seqB +1, number of columns seqA +1
    S = np.zeros((m,n))       # Score matrix of size m x n, initialized to 0
    trace = np.zeros((m,n,2)) # Trace matrix to store direction of optimal move: every matrix needs to store di & dj! vertical and horiontal movement saved!
    S[0,0] = 0.
    trace[0,0,:] = (0.,0.) # backtrack until upper left.
    # Initialize first column: gap penalties for seqA: compares bases in a against gaps in b
    for i in range(1,m): # from 1 to end of seq a
        S[i,0] = i * gap_penalty()
        trace[i,0,:] =(-1,0)  # Came from above (gap in seqB): move down -> up
    # Initialize first row: gap penalties for seqB
    for j in range(1,n):
        S[0,j] = j * gap_penalty()
        trace[0,j,:] =(0,-1)  # Came from left (gap in seqA): move from right to left
    return S,trace

Subsequently, the rest of $S$ is filled as:
$S_{ij}=\max\left\{
\begin{array}{ll}
S_{i-1,j-1} & +d(a_i,b_j)\\
S_{i-1,j} & +d(a_i,-)\\
S_{i,j-1} & +d(-,b_j)
\end{array}
\right.$

This recursion is easily transformed into for-loops. We are free to select the order the matrix is filled in so we select to fill in row per row, i.e. the columns become the inner loop, the rows the outer loop.

Again we keep track of the move that lead to a certain position by filling in the `trace` matrix.

In [ ]:
# Perform global alignment using the Needleman-Wunsch algorithm.
# Returns the aligned sequences (with gaps) and optionally prints the score matrix.
def global_align(seqA,seqB,print_dynamic_matrix = False): # wont print the dynamic matrix, only the final score.
    # Add 1 to lengths for DP matrix (to account for initial gap row/column)
    m, n = len(seqA)+1, len(seqB)+1 # as we added gaps
    S,trace = initiate_global_dp(m,n) # adds 0, -1, -2 etc in first row and column
    # Fill in the rest of the dynamic programming matrix
    for i in range(1,m):
        for j in range(1,n):
            # Calculate scores for match/mismatch, deletion, and insertion
            # Note: i-1 and j-1 because Python uses 0-based indexing
            match = S[i-1,j-1] + match_score(seqA[i-1],seqB[j-1])
            delete = S[i-1,j] + match_score(seqA[i-1],'-')
            insert = S[i,j-1] + match_score('-',seqB[j-1])
            # Choose the maximum score and record the move in trace
            S[i,j] = max(match,delete,insert)
            if match >= max(insert,delete):
                trace[i,j,:] = (-1,-1)  # Diagonal move: match/mismatch
            elif delete >= insert:
                trace[i,j,:] = (-1,0)   # Up: gap in seqB (deletion)
            else:
                trace[i,j,:] = (0,-1)   # Left: gap in seqA (insertion)
    if print_dynamic_matrix:
        print_dynamic(seqA,seqB,S,trace)  # Optional: print the score matrix
    return format_alignment(seqA,seqB,S,trace)  # Traceback to get alignment

Now everything is set. We can try the code for any of our favorite sequences. One can toggle the printout of the dynamic programming matrix by a boolean flag as a third argument.

In [ ]:
# Example: align two short DNA sequences and print the result
seqA,seqB = global_align("GATTA","GCTAC",True)
print_alignment(seqA,seqB)

       -    G    C    T    A    C  
  -    0.0 -2.0 -4.0 -6.0 -8.0-10.0
  G   -2.0  3.0  1.0 -1.0 -3.0 -5.0
  A   -4.0  1.0  2.0  0.0  2.0  0.0
  T   -6.0 -1.0  0.0  5.0  3.0  1.0
  T   -8.0 -3.0 -2.0  3.0  4.0  2.0
  A  -10.0 -5.0 -4.0  1.0  6.0  4.0

Best score: 4.0
GATTA-
G-CTAC


I add a couple of extra alignments, check them manually as an excercise before the exam. Also try to run a couple of them yourself.

In [ ]:
# Another example: align two longer sequences
seqA,seqB = global_align("TGCATTA","GCATTAC",True)
print_alignment(seqA,seqB)

       -    G    C    A    T    T    A    C  
  -    0.0 -2.0 -4.0 -6.0 -8.0-10.0-12.0-14.0
  T   -2.0 -1.0 -3.0 -5.0 -3.0 -5.0 -7.0 -9.0
  G   -4.0  1.0 -1.0 -3.0 -5.0 -4.0 -6.0 -8.0
  C   -6.0 -1.0  4.0  2.0  0.0 -2.0 -4.0 -3.0
  A   -8.0 -3.0  2.0  7.0  5.0  3.0  1.0 -1.0
  T  -10.0 -5.0  0.0  5.0 10.0  8.0  6.0  4.0
  T  -12.0 -7.0 -2.0  3.0  8.0 13.0 11.0  9.0
  A  -14.0 -9.0 -4.0  1.0  6.0 11.0 16.0 14.0

Best score: 14.0
TGCATTA-
-GCATTAC


In [ ]:
# Try aligning even longer sequences for practice
seqA,seqB = global_align("CTATCTCGCTATCCA","CTACGCTATTTCA",True)
print_alignment(seqA,seqB)

       -    C    T    A    C    G    C    T    A    T    T    T    C    A  
  -    0.0 -2.0 -4.0 -6.0 -8.0-10.0-12.0-14.0-16.0-18.0-20.0-22.0-24.0-26.0
  C   -2.0  3.0  1.0 -1.0 -3.0 -5.0 -7.0 -9.0-11.0-13.0-15.0-17.0-19.0-21.0
  T   -4.0  1.0  6.0  4.0  2.0  0.0 -2.0 -4.0 -6.0 -8.0-10.0-12.0-14.0-16.0
  A   -6.0 -1.0  4.0  9.0  7.0  5.0  3.0  1.0 -1.0 -3.0 -5.0 -7.0 -9.0-11.0
  T   -8.0 -3.0  2.0  7.0  8.0  6.0  4.0  6.0  4.0  2.0  0.0 -2.0 -4.0 -6.0
  C  -10.0 -5.0  0.0  5.0 10.0  8.0  9.0  7.0  5.0  3.0  1.0 -1.0  1.0 -1.0
  T  -12.0 -7.0 -2.0  3.0  8.0  9.0  7.0 12.0 10.0  8.0  6.0  4.0  2.0  0.0
  C  -14.0 -9.0 -4.0  1.0  6.0  7.0 12.0 10.0 11.0  9.0  7.0  5.0  7.0  5.0
  G  -16.0-11.0 -6.0 -1.0  4.0  9.0 10.0 11.0  9.0 10.0  8.0  6.0  5.0  6.0
  C  -18.0-13.0 -8.0 -3.0  2.0  7.0 12.0 10.0 10.0  8.0  9.0  7.0  9.0  7.0
  T  -20.0-15.0-10.0 -5.0  0.0  5.0 10.0 15.0 13.0 13.0 11.0 12.0 10.0  8.0
  A  -22.0-17.0-12.0 -7.0 -2.0  3.0  8.0 13.0 18.0 16.0 14.0 12.0 11.0 13.0
  T  -24.0-1